# Forecasting Under Regime Instability: Advanced Model Training and Forecast Evaluation Under Instability

## Purpose


## Imports and Configuration

In [ ]:
%matplotlib inline

from IPython.display import display

import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import statsmodels.api as sm

from pathlib import Path

from scipy.stats import norm
from sklearn.base import clone
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import ElasticNet, HuberRegressor, Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from regime_shift.config import (
    FIG_ROOT as fig_root,
    MODEL_ROOT as model_root,
    PRICE_MODEL_READY_PATH as price_model_ready_path,
    REPORT_ROOT as report_root,
    SAVE_DPI as save_dpi,
    SHOW_DEC as show_dec,
    WAGE_MODEL_READY_PATH as wage_model_ready_path,
)

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 180)
pd.set_option("display.max_rows", 200)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

for path in (fig_root, model_root, report_root):
    Path(path).mkdir(parents=True, exist_ok=True)

sns.set_style("whitegrid")

## Load Wage and Price Dataset

In [ ]:
wage_data = pd.read_csv(wage_model_ready_path)
price_data = pd.read_csv(price_model_ready_path)

for data in [wage_data, price_data]:
    data["date"] = pd.to_datetime(data["date"], errors="coerce")
    data.dropna(subset=["date"], inplace=True)
    data.sort_values("date", inplace=True)
    data.reset_index(drop=True, inplace=True)

## Target Construction and Model Functions

### Define Forecast Targets

In [ ]:
def target_map() -> dict:
    return {
        "wage": [
            "wage_target_3",
            "wage_target_6",
            "wage_target_12",
        ],
        "price": [
            "cpi_target_3",
            "cpi_target_6",
            "cpi_target_12",
            "pce_target_3",
            "pce_target_6",
            "pce_target_12",
        ],
    }

In [ ]:
all_targets = target_map()["wage"] + target_map()["price"]

structural_signal = "log_jolts_ratio"
structural_controls = ["fed_funds", "hy_oas"]

### Define Models

In [ ]:
forecast_models = [
    "Persistence",
    "AR",
    "OLS",
    "Ridge",
    "ElasticNet",
    "Huber",
    "RandomForest",
    "GradientBoosting",
]

In [ ]:
mode_list = ["AR", "Base", "Quits", "Jobless", "Mix", "Curve"]
signal_models = ["OLS", "Ridge", "ElasticNet", "Huber"]

### Horizon and Target Labels

In [ ]:
def horizon(target_col: str) -> int:
    return int(target_col.rsplit("_", 1)[-1])

In [ ]:
def target_name(target_col):
    if target_col.startswith("wage_"):
        return "Wage"
    if target_col.startswith("cpi_"):
        return "CPI"
    if target_col.startswith("pce_"):
        return "PCE"
    raise ValueError("Target column isn't supported.")

In [ ]:
def unique_cols(cols: list[str]) -> list[str]:
    seen = set()
    out = []

    for col in cols:
        if col not in seen:
            seen.add(col)
            out.append(col)

    return out

### Construct Lagged Targets and Feature Blocks

In [ ]:
def add_target_lag(data: pd.DataFrame, target_col: str) -> tuple[pd.DataFrame, str]:
    lag_col = f"{target_col}_lag"
    out = data.copy()
    out[lag_col] = out[target_col].shift(horizon(target_col))

    return out, lag_col

In [ ]:
def base_feature_block(data: pd.DataFrame, target_col: str) -> tuple[pd.DataFrame, list[str], str]:
    out, lag_col = add_target_lag(data, target_col)

    feat_cols = unique_cols(
        [
            structural_signal,
            lag_col,
            *structural_controls,
        ]
    )

    keep_cols = unique_cols(
        [
            "date",
            target_col,
            *feat_cols,
            "pre_regime",
            "shock_regime",
            "post_regime",
        ]
    )

    out = out.loc[:, [col for col in keep_cols if col in out.columns]].copy()
    out = out.dropna().reset_index(drop=True)

    return out, feat_cols, lag_col

In [ ]:
def fetch_base_data(data: pd.DataFrame, target_col: str) -> tuple[pd.DataFrame, list[str], str]:
    return base_feature_block(data, target_col)

In [ ]:
def fetch_data(data: pd.DataFrame, target_col: str) -> pd.DataFrame:
    sample, feat_cols, lag_col = fetch_base_data(data, target_col)

    keep_cols = unique_cols(
        [
            "date",
            target_col,
            lag_col,
            *feat_cols,
            "pre_regime",
            "shock_regime",
            "post_regime",
        ]
    )

    return sample.loc[:, keep_cols].copy()

In [ ]:
def sample_summary(data: pd.DataFrame, target_col: str) -> pd.DataFrame:
    sample = fetch_data(data, target_col)

    return pd.DataFrame(
        {
            "target": [target_name(target_col)],
            "target_col": [target_col],
            "horizon": [horizon(target_col)],
            "rows": [len(sample)],
            "start": [sample["date"].min()],
            "end": [sample["date"].max()],
            "pre_rows": [int(sample["pre_regime"].sum())],
            "shock_rows": [int(sample["shock_regime"].sum())],
            "post_rows": [int(sample["post_regime"].sum())],
        }
    )

In [ ]:
sample_parts = []

for target_col in all_targets:
    base_data = wage_data if target_col.startswith("wage_") else price_data
    sample_parts.append(sample_summary(base_data, target_col))

sample_data = pd.concat(sample_parts, ignore_index=True)
display(sample_data.sort_values(["target", "horizon"]).reset_index(drop=True))

## Forecast Evaluation Metrics

In [ ]:
def root_mean_square_error(actual: np.ndarray, pred: np.ndarray) -> float:
    return float(np.sqrt(mean_squared_error(actual, pred)))


def mean_absolute_error_value(actual: np.ndarray, pred: np.ndarray) -> float:
    return float(mean_absolute_error(actual, pred))


def out_sample_r_squared(actual: np.ndarray, pred: np.ndarray, base_pred: np.ndarray) -> float:
    model_mse = mean_squared_error(actual, pred)
    base_mse = mean_squared_error(actual, base_pred)

    if base_mse == 0:
        return np.nan

    return float(1.0 - (model_mse / base_mse))


def score_row(
    target_col: str,
    model: str,
    split: str,
    actual: np.ndarray,
    pred: np.ndarray,
    base_pred: np.ndarray,
    train_end: pd.Timestamp,
    test_start: pd.Timestamp,
    test_end: pd.Timestamp,
) -> dict:
    return {
        "target": target_name(target_col),
        "target_col": target_col,
        "horizon": horizon(target_col),
        "model": model,
        "split": split,
        "rmse": root_mean_square_error(actual, pred),
        "mae": mean_absolute_error_value(actual, pred),
        "oos_r2_vs_persistence": out_sample_r_squared(actual, pred, base_pred),
        "row_count": len(actual),
        "train_end": train_end,
        "test_start": test_start,
        "test_end": test_end,
    }

##  Modeling  Set-Up

In [ ]:
def fit_persistence(train_target: pd.Series, test_feat: pd.DataFrame) -> np.ndarray:
    last_value = float(train_target.iloc[-1])
    return np.repeat(last_value, len(test_feat))


def fit_ols(train_feat: pd.DataFrame, train_target: pd.Series, test_feat: pd.DataFrame) -> np.ndarray:
    train_data = sm.add_constant(train_feat, has_constant="add")
    test_data = sm.add_constant(test_feat, has_constant="add")
    test_data = test_data.reindex(columns=train_data.columns, fill_value=0.0)

    fit = sm.OLS(train_target, train_data).fit()
    pred = fit.predict(test_data)

    return np.asarray(pred)


def fit_pipe(model, train_feat: pd.DataFrame, train_target: pd.Series, test_feat: pd.DataFrame) -> np.ndarray:
    pipe = Pipeline(
        [
            ("scale", StandardScaler()),
            ("model", clone(model)),
        ]
    )

    pipe.fit(train_feat, train_target)
    pred = pipe.predict(test_feat)

    return np.asarray(pred)


def fit_tree(model, train_feat: pd.DataFrame, train_target: pd.Series, test_feat: pd.DataFrame) -> np.ndarray:
    fit = clone(model)
    fit.fit(train_feat, train_target)
    pred = fit.predict(test_feat)

    return np.asarray(pred)

In [ ]:
linear_model_map = {
    "Ridge": Ridge(alpha=1.0),
    # "Lasso": Lasso(alpha=0.02, max_iter=20000, random_state=42),
    "ElasticNet": ElasticNet(alpha=0.03, l1_ratio=0.5, max_iter=20000, random_state=42),
    "Huber": HuberRegressor(epsilon=1.35, alpha=0.0001, max_iter=500),
}

In [ ]:
tree_model_map = {
    "RandomForest": RandomForestRegressor(
        n_estimators=400,
        max_depth=4,
        min_samples_leaf=5,
        random_state=42,
    ),
    "GradientBoosting": GradientBoostingRegressor(
        n_estimators=250,
        learning_rate=0.03,
        max_depth=2,
        subsample=0.90,
        random_state=42,
    ),
}

In [ ]:
def model_pred(
    model: str,
    train_feat: pd.DataFrame,
    train_target: pd.Series,
    test_feat: pd.DataFrame,
    lag_col: str,
) -> np.ndarray:
    if model == "Persistence":
        return fit_persistence(train_target, test_feat)

    if model == "AR":
        return fit_ols(train_feat[[lag_col]], train_target, test_feat[[lag_col]])

    if model == "OLS":
        return fit_ols(train_feat, train_target, test_feat)

    if model in linear_model_map:
        return fit_pipe(linear_model_map[model], train_feat, train_target, test_feat)

    if model in tree_model_map:
        return fit_tree(tree_model_map[model], train_feat, train_target, test_feat)

    raise ValueError(f"Unknown model: {model}")

In [ ]:
def fold_points(
    data: pd.DataFrame,
    folds: int = 5,
    min_train: int = 60,
) -> list[tuple[slice, slice]]:
    row_count = len(data)

    if row_count <= min_train + 1:
        return []

    test_size = max(1, row_count // (folds + 1))
    points = []
    train_end = min_train

    while train_end + test_size <= row_count and len(points) < folds:
        points.append((slice(0, train_end), slice(train_end, train_end + test_size)))
        train_end += test_size

    return points

In [ ]:
def pre_post_run(data: pd.DataFrame, target_col: str) -> tuple[pd.DataFrame, pd.DataFrame]:
    sample, feat_cols, lag_col = fetch_base_data(data, target_col)

    train_data = sample.loc[sample["pre_regime"].eq(1)].copy()
    test_data = sample.loc[sample["post_regime"].eq(1)].copy()

    if train_data.empty or test_data.empty:
        return pd.DataFrame(), pd.DataFrame()

    train_feat = train_data[feat_cols].copy()
    train_target = train_data[target_col].copy()
    test_feat = test_data[feat_cols].copy()
    test_target = test_data[target_col].copy()

    base_pred = fit_persistence(train_target, test_feat)

    score_parts = []
    prediction_parts = []

    for model in forecast_models:
        pred = model_pred(model, train_feat, train_target, test_feat, lag_col)

        score_parts.append(
            score_row(
                target_col=target_col,
                model=model,
                split="Pre to Post",
                actual=np.asarray(test_target),
                pred=np.asarray(pred),
                base_pred=np.asarray(base_pred),
                train_end=train_data["date"].max(),
                test_start=test_data["date"].min(),
                test_end=test_data["date"].max(),
            )
        )

        prediction_parts.append(
            pd.DataFrame(
                {
                    "date": test_data["date"].values,
                    "actual": np.asarray(test_target),
                    "prediction": np.asarray(pred),
                    "benchmark": np.asarray(base_pred),
                    "model": model,
                    "target": target_name(target_col),
                    "target_col": target_col,
                    "horizon": horizon(target_col),
                    "split": "Pre to Post",
                }
            )
        )

    score_data = pd.DataFrame(score_parts)
    prediction_data = pd.concat(prediction_parts, ignore_index=True)

    return score_data, prediction_data

In [ ]:
def window_run(
    data: pd.DataFrame,
    target_cols: list[str],
    folds: int = 5,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    score_parts = []
    prediction_parts = []

    for target_col in target_cols:
        sample, feat_cols, lag_col = fetch_base_data(data, target_col)

        points = fold_points(
            sample,
            folds=folds,
            min_train=max(48, horizon(target_col) * 8),
        )

        for train_idx, test_idx in points:
            train_data = sample.iloc[train_idx].copy()
            test_data = sample.iloc[test_idx].copy()

            if train_data.empty or test_data.empty:
                continue

            train_feat = train_data[feat_cols].copy()
            train_target = train_data[target_col].copy()
            test_feat = test_data[feat_cols].copy()
            test_target = test_data[target_col].copy()

            base_pred = fit_persistence(train_target, test_feat)

            for model in forecast_models:
                pred = model_pred(model, train_feat, train_target, test_feat, lag_col)

                score_parts.append(
                    score_row(
                        target_col=target_col,
                        model=model,
                        split="Expanding Window",
                        actual=np.asarray(test_target),
                        pred=np.asarray(pred),
                        base_pred=np.asarray(base_pred),
                        train_end=train_data["date"].max(),
                        test_start=test_data["date"].min(),
                        test_end=test_data["date"].max(),
                    )
                )

                prediction_parts.append(
                    pd.DataFrame(
                        {
                            "date": test_data["date"].values,
                            "actual": np.asarray(test_target),
                            "prediction": np.asarray(pred),
                            "benchmark": np.asarray(base_pred),
                            "model": model,
                            "target": target_name(target_col),
                            "target_col": target_col,
                            "horizon": horizon(target_col),
                            "split": "Expanding Window",
                        }
                    )
                )

    score_data = pd.DataFrame(score_parts)
    prediction_data = pd.concat(prediction_parts, ignore_index=True) if prediction_parts else pd.DataFrame()

    return score_data, prediction_data

In [ ]:
split_score_parts = []
split_prediction_parts = []

for target_col in all_targets:
    base_data = wage_data if target_col.startswith("wage_") else price_data
    score_data, prediction_data = pre_post_run(base_data, target_col)

    if not score_data.empty:
        split_score_parts.append(score_data)

    if not prediction_data.empty:
        split_prediction_parts.append(prediction_data)

split_scores = pd.concat(split_score_parts, ignore_index=True) if split_score_parts else pd.DataFrame()
split_predictions = pd.concat(split_prediction_parts, ignore_index=True) if split_prediction_parts else pd.DataFrame()


window_wage_scores, window_wage_predictions = window_run(wage_data, target_map()["wage"])
window_price_scores, window_price_predictions = window_run(price_data, target_map()["price"])

window_scores = pd.concat(
    [
        window_wage_scores,
        window_price_scores,
    ],
    ignore_index=True,
) if (not window_wage_scores.empty or not window_price_scores.empty) else pd.DataFrame()

window_predictions = pd.concat(
    [
        window_wage_predictions,
        window_price_predictions,
    ],
    ignore_index=True,
) if (not window_wage_predictions.empty or not window_price_predictions.empty) else pd.DataFrame()

display(split_scores.sort_values(["target", "horizon", "rmse", "mae"]).reset_index(drop=True).round(show_dec))
display(window_scores.sort_values(["target", "horizon", "rmse", "mae"]).reset_index(drop=True).round(show_dec))

In [ ]:
all_scores = pd.concat(
    [
        split_scores,
        window_scores,
    ],
    ignore_index=True,
) if (not split_scores.empty or not window_scores.empty) else pd.DataFrame()

score_summary = (
    all_scores.groupby(
        ["target", "target_col", "horizon", "model", "split"],
        as_index=False,
    )
    .agg(
        rmse=("rmse", "mean"),
        mae=("mae", "mean"),
        oos_r2=("oos_r2_vs_persistence", "mean"),
        row_count=("row_count", "sum"),
    )
    .sort_values(["split", "target", "horizon", "rmse", "mae"])
    .reset_index(drop=True)
)

rank_summary = score_summary.copy()
rank_summary["rmse_rank"] = rank_summary.groupby(["split", "target_col"])["rmse"].rank(method="dense")
rank_summary["mae_rank"] = rank_summary.groupby(["split", "target_col"])["mae"].rank(method="dense")
rank_summary["avg_rank"] = (rank_summary["rmse_rank"] + rank_summary["mae_rank"]) / 2.0

best_summary = (
    rank_summary.sort_values(["split", "target_col", "avg_rank", "rmse", "mae"])
    .groupby(["split", "target_col"], as_index=False)
    .first()
    .sort_values(["split", "target", "horizon"])
    .reset_index(drop=True)
)

display(score_summary.round(show_dec))
display(best_summary.round(show_dec))

In [ ]:
core_summary = score_summary.loc[
    score_summary["model"].isin(["AR", "OLS", "Ridge", "ElasticNet", "Huber"])
].copy()

ar_summary = core_summary.loc[core_summary["model"].eq("AR")].copy()
ar_summary = ar_summary.rename(
    columns={
        "rmse": "ar_rmse",
        "mae": "ar_mae",
        "oos_r2": "ar_oos_r2",
    }
)[["target_col", "split", "ar_rmse", "ar_mae", "ar_oos_r2"]]

best_core = (
    core_summary.sort_values(["split", "target_col", "rmse", "mae"])
    .groupby(["split", "target_col"], as_index=False)
    .first()
    .sort_values(["split", "target", "horizon"])
    .reset_index(drop=True)
)

gain_summary = best_core.merge(ar_summary, on=["target_col", "split"], how="left")
gain_summary["rmse_gain_vs_ar"] = gain_summary["ar_rmse"] - gain_summary["rmse"]
gain_summary["mae_gain_vs_ar"] = gain_summary["ar_mae"] - gain_summary["mae"]

display(
    gain_summary[
        [
            "target",
            "target_col",
            "horizon",
            "split",
            "model",
            "rmse",
            "mae",
            "oos_r2",
            "ar_rmse",
            "ar_mae",
            "rmse_gain_vs_ar",
            "mae_gain_vs_ar",
        ]
    ].round(show_dec)
)

In [ ]:
def signal_feature_block(
    data: pd.DataFrame,
    target_col: str,
    mode: str,
) -> tuple[pd.DataFrame, list[str]]:
    out, lag_col = add_target_lag(data, target_col)

    if mode == "AR":
        feat_cols = [lag_col]

    elif mode == "Base":
        feat_cols = ["log_jolts_ratio", lag_col, "fed_funds", "hy_oas"]

    elif mode == "Quits":
        feat_cols = ["quits_rate", lag_col, "fed_funds", "hy_oas"]

    elif mode == "Jobless":
        feat_cols = ["unemployment_rate", lag_col, "fed_funds", "hy_oas"]

    elif mode == "Mix":
        feat_cols = [
            "log_jolts_ratio",
            "quits_rate",
            "unemployment_rate",
            lag_col,
            "fed_funds",
            "hy_oas",
        ]

    elif mode == "Curve":
        square_col = "log_jolts_ratio_square"
        out[square_col] = out["log_jolts_ratio"] ** 2
        feat_cols = ["log_jolts_ratio", square_col, lag_col, "fed_funds", "hy_oas"]

    else:
        raise ValueError("Mode is not supported.")

    feat_cols = [col for col in feat_cols if col in out.columns]

    return out, feat_cols


def keep_data(
    data: pd.DataFrame,
    target_col: str,
    feat_cols: list[str],
    drop_shock: bool = True,
) -> pd.DataFrame:
    keep_cols = unique_cols(
        [
            "date",
            target_col,
            *feat_cols,
            "pre_regime",
            "shock_regime",
            "post_regime",
        ]
    )

    sample = data.loc[:, [col for col in keep_cols if col in data.columns]].copy()

    for col in sample.columns:
        if col != "date":
            sample[col] = pd.to_numeric(sample[col], errors="coerce")

    sample = sample.dropna().reset_index(drop=True)

    if drop_shock and "shock_regime" in sample.columns:
        sample = sample.loc[sample["shock_regime"].eq(0)].copy().reset_index(drop=True)

    return sample

In [ ]:
def fit_signal_model(
    model: str,
    train_feat: pd.DataFrame,
    train_target: pd.Series,
    test_feat: pd.DataFrame,
) -> np.ndarray:
    if model == "OLS":
        return fit_ols(train_feat, train_target, test_feat)

    if model in linear_model_map:
        return fit_pipe(linear_model_map[model], train_feat, train_target, test_feat)

    raise ValueError(f"Unknown signal model: {model}")

In [ ]:
def post_run(data: pd.DataFrame, target_col: str, mode: str, folds: int = 4) -> tuple[pd.DataFrame, pd.DataFrame]:
    out, feat_cols = signal_feature_block(data, target_col, mode)
    sample = keep_data(out, target_col, feat_cols, drop_shock=True)

    if "post_regime" not in sample.columns:
        return pd.DataFrame(), pd.DataFrame()

    sample = sample.loc[sample["post_regime"].eq(1)].copy().reset_index(drop=True)

    if mode == "Curve" and target_name(target_col) == "Wage":
        return pd.DataFrame(), pd.DataFrame()

    if sample.empty:
        return pd.DataFrame(), pd.DataFrame()

    min_train = max(8, horizon(target_col))
    points = fold_points(sample, folds=folds, min_train=min_train)

    if not points and len(sample) >= 6:
        split_at = max(4, len(sample) // 2)

        if split_at < len(sample):
            points = [(slice(0, split_at), slice(split_at, len(sample)))]

    if not points:
        return pd.DataFrame(), pd.DataFrame()

    score_parts = []
    pred_parts = []

    if mode == "AR":
        post_models = ["AR"]
    else:
        post_models = signal_models

    for train_slice, test_slice in points:
        train_data = sample.iloc[train_slice].copy()
        test_data = sample.iloc[test_slice].copy()

        if train_data.empty or test_data.empty:
            continue

        train_feat = train_data[feat_cols].copy()
        train_target = train_data[target_col].copy()
        test_feat = test_data[feat_cols].copy()
        test_target = test_data[target_col].copy()

        base_pred = fit_persistence(train_target, test_feat)

        for model in post_models:
            if mode == "AR" and model == "AR":
                pred = fit_ols(train_feat, train_target, test_feat)
            else:
                pred = fit_signal_model(model, train_feat, train_target, test_feat)

            score_parts.append(
                score_row(
                    target_col=target_col,
                    model=model,
                    split="Post Window",
                    actual=np.asarray(test_target),
                    pred=np.asarray(pred),
                    base_pred=np.asarray(base_pred),
                    train_end=train_data["date"].max(),
                    test_start=test_data["date"].min(),
                    test_end=test_data["date"].max(),
                ) | {"mode": mode}
            )

            pred_parts.append(
                pd.DataFrame(
                    {
                        "date": test_data["date"].values,
                        "actual": np.asarray(test_target),
                        "prediction": np.asarray(pred),
                        "benchmark": np.asarray(base_pred),
                        "model": model,
                        "mode": mode,
                        "target": target_name(target_col),
                        "target_col": target_col,
                        "horizon": horizon(target_col),
                        "split": "Post Window",
                    }
                )
            )

    score_data = pd.DataFrame(score_parts)
    pred_data = pd.concat(pred_parts, ignore_index=True) if pred_parts else pd.DataFrame()

    return score_data, pred_data

In [ ]:
def clark_west_value(
    error_base: np.ndarray,
    error_test: np.ndarray,
    pred_base: np.ndarray,
    pred_test: np.ndarray,
) -> tuple[float, float]:
    adjust = (error_base ** 2) - ((error_test ** 2) - ((pred_base - pred_test) ** 2))
    adjust = pd.Series(adjust).dropna()

    if len(adjust) < 8:
        return np.nan, np.nan

    mean_value = adjust.mean()
    std_value = adjust.std(ddof=1)

    if pd.isna(std_value) or std_value == 0:
        return np.nan, np.nan

    stat = float(np.sqrt(len(adjust)) * mean_value / std_value)
    p_value = float(1 - norm.cdf(stat))

    return stat, p_value


def clark_west_row(pred_data: pd.DataFrame, target_col: str, model: str, mode: str) -> pd.DataFrame:
    base = pred_data.loc[
        pred_data["mode"].eq("Base") & pred_data["model"].eq(model),
        ["date", "actual", "prediction"],
    ].rename(
        columns={
            "actual": "actual_base",
            "prediction": "pred_base",
        }
    )

    test = pred_data.loc[
        pred_data["mode"].eq(mode) & pred_data["model"].eq(model),
        ["date", "actual", "prediction"],
    ].rename(
        columns={
            "actual": "actual_test",
            "prediction": "pred_test",
        }
    )

    join = base.merge(test, on="date", how="inner")

    if join.empty:
        return pd.DataFrame(
            {
                "target": [target_name(target_col)],
                "target_col": [target_col],
                "horizon": [horizon(target_col)],
                "model": [model],
                "mode": [mode],
                "clark_west": [np.nan],
                "p_value": [np.nan],
                "rows": [0],
            }
        )

    error_base = join["actual_base"].to_numpy() - join["pred_base"].to_numpy()
    error_test = join["actual_test"].to_numpy() - join["pred_test"].to_numpy()

    stat, p_value = clark_west_value(
        error_base=error_base,
        error_test=error_test,
        pred_base=join["pred_base"].to_numpy(),
        pred_test=join["pred_test"].to_numpy(),
    )

    return pd.DataFrame(
        {
            "target": [target_name(target_col)],
            "target_col": [target_col],
            "horizon": [horizon(target_col)],
            "model": [model],
            "mode": [mode],
            "clark_west": [stat],
            "p_value": [p_value],
            "rows": [len(join)],
        }
    )

In [ ]:
post_parts = []
pred_parts = []
clark_parts = []

for target_col in all_targets:
    base_data = wage_data if target_col.startswith("wage_") else price_data

    for mode in mode_list:
        score_data, pred_data = post_run(base_data, target_col, mode)

        if not score_data.empty:
            post_parts.append(score_data)

        if not pred_data.empty:
            pred_parts.append(pred_data)

post_scores = pd.concat(post_parts, ignore_index=True) if post_parts else pd.DataFrame()
post_preds = pd.concat(pred_parts, ignore_index=True) if pred_parts else pd.DataFrame()

if not post_preds.empty:
    for target_col in post_preds["target_col"].drop_duplicates():
        target_pred = post_preds.loc[post_preds["target_col"].eq(target_col)].copy()

        for model in signal_models:
            for mode in ["Quits", "Jobless", "Mix", "Curve"]:
                part = clark_west_row(target_pred, target_col, model, mode)
                clark_parts.append(part)

clark_data = pd.concat(clark_parts, ignore_index=True) if clark_parts else pd.DataFrame()

if not post_scores.empty:
    post_table = (
        post_scores.groupby(
            ["target", "target_col", "horizon", "mode", "model", "split"],
            as_index=False,
        )
        .agg(
            rmse=("rmse", "mean"),
            mae=("mae", "mean"),
            oos_r2=("oos_r2_vs_persistence", "mean"),
            row_count=("row_count", "sum"),
        )
        .sort_values(["target", "horizon", "rmse", "mae"])
        .reset_index(drop=True)
    )
else:
    post_table = pd.DataFrame(
        columns=["target", "target_col", "horizon", "mode", "model", "split", "rmse", "mae", "oos_r2", "row_count"]
    )

display(post_table.round(show_dec))
display(clark_data.round(show_dec))

In [ ]:
signal_best = (
    post_table.sort_values(["target_col", "model", "rmse", "mae"])
    .groupby(["target_col", "model"], as_index=False)
    .first()
    .sort_values(["target", "horizon", "model"])
    .reset_index(drop=True)
) if not post_table.empty else pd.DataFrame()

signal_win = (
    signal_best.groupby(["mode"], as_index=False)
    .agg(
        win_count=("target_col", "count"),
        mean_rmse=("rmse", "mean"),
        mean_mae=("mae", "mean"),
        mean_oos_r2=("oos_r2", "mean"),
    )
    .sort_values(["win_count", "mean_rmse"], ascending=[False, True])
    .reset_index(drop=True)
) if not signal_best.empty else pd.DataFrame()

display(signal_best.round(show_dec))
display(signal_win.round(show_dec))

In [ ]:
final_table = pd.concat(
    [
        score_summary,
        post_table,
    ],
    ignore_index=True,
) if (not score_summary.empty or not post_table.empty) else pd.DataFrame()

final_table = final_table.sort_values(["split", "target", "horizon", "rmse", "mae"]).reset_index(drop=True)
display(final_table.round(show_dec))

In [ ]:
forecast_plot_data = score_summary.copy()

fig, axes = plt.subplots(1, 2, figsize=(18, 7), sharey=False)

for ax, split_name in zip(axes, ["Pre to Post", "Expanding Window"]):
    split_data = forecast_plot_data.loc[forecast_plot_data["split"].eq(split_name)].copy()

    if split_data.empty:
        ax.set_axis_off()
        continue

    split_data = split_data.sort_values(["target", "model", "horizon"])

    sns.lineplot(
        data=split_data,
        x="horizon",
        y="rmse",
        hue="model",
        style="target",
        markers=True,
        dashes=False,
        linewidth=2.2,
        markersize=7,
        ax=ax,
    )

    ax.set_title(split_name, pad=10, fontsize=14, weight="bold")
    ax.set_xlabel("Forecast Horizon (Months)", labelpad=8)
    ax.set_ylabel("Root Mean Square Error", labelpad=8)
    ax.set_xticks(sorted(split_data["horizon"].unique()))
    ax.grid(True, alpha=0.25)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

fig.tight_layout()
fig.savefig(fig_root / "05_forecast_benchmark_errors.png", dpi=save_dpi, bbox_inches="tight")
plt.show()

In [ ]:
plot_post = post_table.copy()

fig, ax = plt.subplots(figsize=(12, 7))

if not plot_post.empty:
    plot_post["label"] = plot_post["mode"] + " | " + plot_post["model"]

    sns.lineplot(
        data=plot_post,
        x="horizon",
        y="rmse",
        hue="label",
        style="target",
        markers=True,
        dashes=False,
        linewidth=2.2,
        ax=ax,
    )

ax.set_title("Post Break Signal Errors")
ax.set_xlabel("Forecast Horizon (Months)")
ax.set_ylabel("Root Mean Square Error")

if not plot_post.empty:
    ax.set_xticks(sorted(plot_post["horizon"].dropna().unique()))

ax.legend(frameon=True)

fig.tight_layout()
fig.savefig(fig_root / "05_forecast_benchmark_post_signal_errors.png", dpi=save_dpi, bbox_inches="tight")
plt.show()

In [ ]:
score_summary.to_csv(report_root / "05_forecast_benchmark_summary.csv", index=False)
best_summary.to_csv(report_root / "05_forecast_benchmark_best_models.csv", index=False)
gain_summary.to_csv(report_root / "05_forecast_benchmark_gain_vs_ar.csv", index=False)

split_scores.to_csv(report_root / "05_forecast_benchmark_pre_post_scores.csv", index=False)
window_scores.to_csv(report_root / "05_forecast_benchmark_window_scores.csv", index=False)
post_table.to_csv(report_root / "05_forecast_benchmark_post_scores.csv", index=False)

signal_best.to_csv(report_root / "05_forecast_benchmark_signal_best.csv", index=False)
signal_win.to_csv(report_root / "05_forecast_benchmark_signal_wins.csv", index=False)
clark_data.to_csv(report_root / "05_forecast_benchmark_clark_west.csv", index=False)

split_predictions.to_csv(model_root / "05_forecast_benchmark_pre_post_predictions.csv", index=False)
window_predictions.to_csv(model_root / "05_forecast_benchmark_window_predictions.csv", index=False)
post_preds.to_csv(model_root / "05_forecast_benchmark_post_predictions.csv", index=False)

In [ ]:
display(best_summary.round(show_dec))
display(gain_summary.round(show_dec))
display(signal_best.round(show_dec))
display(signal_win.round(show_dec))
display(clark_data.round(show_dec))

## Conclusion and Findings